# Setup

In [1]:
# Select the number of records to process in the Main Crawl section
# Set the number of parallel workers according to the available CPU cores and memory

In [2]:
!pip install -r requirements.txt --disable-pip-version-check | tail -n 5

In [3]:
import base64
import hashlib
import os
import re
import shutil
import time
from datetime import date
from pathlib import Path

import pandas as pd
from pandarallel import pandarallel
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

In [4]:
# stop selenium messages
os.environ["SE_AVOID_STATS"] = "true"

## Dirs

In [5]:
dirs = [
    "__screens/all",
    "__screens/structural-candidate-yes",
    "__screens/structural-candidate-no",
]

shutil.rmtree("__screens", ignore_errors=True)

for item in dirs:
    path = Path(item)
    path.mkdir(parents=True, exist_ok=True)

## Helpers

In [6]:
def filename_url(url,phish_id):
    url = str(url)
    h = hashlib.sha256(url.encode("utf-8")).hexdigest()[:12]
    name = re.sub(r"[^a-zA-Z0-9._-]+", "_", str(url))[:120]
    return f"{name}_{phish_id}_{h}.png"

assert filename_url(
    'https://www.fekt.vut.cz/o_fakulte/aktualita/338428',1) == 'https_www.fekt.vut.cz_o_fakulte_aktualita_338428_1_671d8f473f33.png'

In [7]:
def copy_screen_meta(r, target_directory):

    if pd.isna(r.get("crawl_screen", pd.NA)):
        return pd.NA

    screen_filename = r["crawl_screen"]

    src_screen_path = os.path.join(
        "__screens/all",
        screen_filename,
    )

    dst_screen_path = os.path.join(
        target_directory,
        screen_filename,
    )

    try:
        shutil.copy2(src_screen_path, dst_screen_path)
    except Exception:
        return pd.NA

    # metadata
    meta_columns = [
        "phishtank_id",
        "phishtank_url",
        "phishtank_online",
        "phishtank_verified",
        "crawl_date",
        "crawl_final_url",
        "crawl_screen",
        "crawl_page_input",
        "crawl_page_text",
    ]

    base_name = os.path.splitext(screen_filename)[0]
    meta_filename = base_name + ".csv"

    dst_meta_path = os.path.join(
        target_directory,
        meta_filename,
    )

    try:
        r[meta_columns].to_frame().T.to_csv(
            dst_meta_path,
            index=False,
            header=False,
        )
    except Exception:
        return pd.NA

    return meta_filename

# Original PhishTank dataset

In [8]:
######################################
# PhishTank dataset date 2026-07-06
######################################
data = pd.read_csv("dataset-original/original-online.csv")
data.pop('target')
data.pop('submission_time')
data.pop('verification_time')
data.pop('phish_detail_url')

data.online = data.online.replace({"yes": True, "no": False}).astype("boolean")
data.verified = data.verified.replace({"yes": True, "no": False}).astype("boolean")

todrop = data.loc[data.online != True]
display(len(todrop))
data = data.drop(todrop.index)

todrop = data.loc[data.verified != True]
display(len(todrop))
data = data.drop(todrop.index)

todrop = data[data.duplicated(subset='url')]
display(len(todrop))
data = data.drop(todrop.index)

data = data.rename(
    columns={
        "phish_id": "phishtank_id",
        "url": "phishtank_url",
        "verified": "phishtank_verified",
        "online": "phishtank_online",        
    }
)

data

0

0

6

,phishtank_id,phishtank_url,phishtank_verified,phishtank_online
0,9470895,http://allegrolokalnie.ofera-lokalna62167981.shop,True,True
1,9470893,https://suscribe-prime.info/,True,True
2,9470892,https://suscribe-prime.info/index.php,True,True
3,9470891,https://allegrolokalnie.lokalna-ofeta76546.cfd...,True,True
4,9470886,https://allegrolokalnie.pl-oferta71028169.shop,True,True
...,...,...,...,...
65132,4173961,http://webmailadmin0.myfreesites.net/,True,True
65133,2042606,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...,True,True
65134,1865971,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...,True,True
65135,1460953,http://www.habbocreditosparati.blogspot.com/,True,True


In [9]:
# Create a reproducible sample for manual PhishTank validity assessment
# data_check = data.sample(1000,random_state=42)
# data_check['manual_available'] = "TBD"
# data_check['manual_data_harvesting_phishing_confirmed'] = "TBD"
# data_check.to_csv('dataset-phishtank-sample/phishtank-sample-TBD.csv',index=False)
# data_check.phishtank_url.to_csv('dataset-phishtank-sample/phishtank-sample-url.csv',index=False,header=None)
# data_check

# Crawl PhishTank-reported webpages

## Helpers

In [10]:
def save_screenshot(driver, path):
    
    driver.execute_script("""
        const el =
            document.scrollingElement ||
            document.documentElement ||
            document.body;

        if (el) {
            window.scrollTo(0, el.scrollHeight);
        }
    """)
    
    time.sleep(10)

    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(5)

    metrics = driver.execute_cdp_cmd("Page.getLayoutMetrics", {})
    content_size = metrics["contentSize"]

    width = int(content_size["width"])
    height = int(content_size["height"])

    height = min(height, 30000)
    width = min(width, 4096)

    # Fall back to a viewport screenshot when the requested full-page image exceeds 60 million pixels
    if width * height > 60_000_000:
        driver.save_screenshot(path)
        return

    screenshot = driver.execute_cdp_cmd(
        "Page.captureScreenshot",
        {
            "format": "png",
            "fromSurface": True,
            "captureBeyondViewport": True,
            "clip": {
                "x": 0,
                "y": 0,
                "width": width,
                "height": height,
                "scale": 1,
            },
        },
    )

    with open(path, "wb") as f:
        f.write(base64.b64decode(screenshot["data"]))

In [11]:
PAGE_SCRIPT = """
const data = {
    text: "",
    input: []
};

if (document.body) {
    data.text = (document.body.innerText || "").slice(0, 300000);
}

function getPageGeometry() {
    const doc = document.documentElement;
    const body = document.body;

    const width = Math.max(
        doc ? doc.scrollWidth : 0,
        doc ? doc.clientWidth : 0,
        body ? body.scrollWidth : 0,
        body ? body.clientWidth : 0,
        window.innerWidth || 0
    );

    const height = Math.max(
        doc ? doc.scrollHeight : 0,
        doc ? doc.clientHeight : 0,
        body ? body.scrollHeight : 0,
        body ? body.clientHeight : 0,
        window.innerHeight || 0
    );

    return {
        width: width,
        height: height
    };
}

function isInsidePageGeometry(rect, page) {
    const x = rect.left + window.scrollX;
    const y = rect.top + window.scrollY;

    const right = x + rect.width;
    const bottom = y + rect.height;

    return (
        x >= 0 &&
        y >= 0 &&
        right <= page.width &&
        bottom <= page.height
    );
}

function isVisible(el, page) {
    if (!el.isConnected) {
        return false;
    }

    const type = (el.getAttribute("type") || "text").toLowerCase();

    if (type === "hidden") {
        return false;
    }

    const rect = el.getBoundingClientRect();

    if (rect.width <= 0 || rect.height <= 0) {
        return false;
    }

    if (!isInsidePageGeometry(rect, page)) {
        return false;
    }

    let node = el;

    while (node && node.nodeType === 1) {
        const style = window.getComputedStyle(node);

        if (style.display === "none") {
            return false;
        }

        if (style.visibility === "hidden" || style.visibility === "collapse") {
            return false;
        }

        if (parseFloat(style.opacity) === 0) {
            return false;
        }

        if (node.hidden) {
            return false;
        }

        if (node.getAttribute("aria-hidden") === "true") {
            return false;
        }

        node = node.parentElement;
    }

    return true;
}

const page = getPageGeometry();

for (const el of document.querySelectorAll("input")) {
    if (!isVisible(el, page)) {
        continue;
    }

    const rect = el.getBoundingClientRect();

    const x = rect.left + window.scrollX;
    const y = rect.top + window.scrollY;

    data.input.push({
        tag: el.tagName.toLowerCase(),
        type: (el.getAttribute("type") || "text").toLowerCase(),

        visible: true,

        width: rect.width,
        height: rect.height,

        x: x,
        y: y,

        page_width: page.width,
        page_height: page.height,

        disabled: el.disabled || false,
        readonly: el.readOnly || false,

        placeholder: el.getAttribute("placeholder") || "",
        name: el.getAttribute("name") || "",
        id: el.getAttribute("id") || "",
        autocomplete: el.getAttribute("autocomplete") || ""
    });
}

return data;
"""

In [12]:
def make_driver():
    chrome_options = Options()

    chrome_options.page_load_strategy = "none"
    
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=2560,1440")
    chrome_options.add_argument(
        "--user-agent=Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)

    driver.set_page_load_timeout(30)
    driver.set_script_timeout(30)

    return driver

In [13]:
def page_has_content(driver):
    try:
        text_len = driver.execute_script("""
            const text = document.body ? (document.body.innerText || "") : "";
            return text.trim().length;
        """)

        return text_len > 0
        
    except Exception:
        return False

In [14]:
RESULT_COLUMNS = [
    "crawl_final_url",
    "crawl_page_input",
    "crawl_page_text",
    "crawl_error",
    "crawl_date",
    "crawl_screen",
]


def empty_result():
    return {
        "crawl_final_url": pd.NA,
        "crawl_page_input": pd.NA,
        "crawl_page_text": pd.NA,
        "crawl_error": pd.NA,
        "crawl_date": date.today().isoformat(),
        "crawl_screen": pd.NA,
    }


def result_series(result):
    return pd.Series(result, index=RESULT_COLUMNS)


def check_url(url, phish_id):
    result = empty_result()
    errors = []
    driver = None

    try:
        driver = make_driver()

        driver.get(url)
        time.sleep(45)

        if not page_has_content(driver):
            result["crawl_error"] = "no_rendered_text"
            return result_series(result)

        result["crawl_final_url"] = url

        try:
            browser_url = driver.current_url

            if browser_url and not browser_url.startswith("blob:"):
                result["crawl_final_url"] = browser_url

        except Exception as e:
            errors.append(f"final_url_error: {repr(e)}")

        try:
            filename = filename_url(result["crawl_final_url"], phish_id)

            save_screenshot(
                driver,
                os.path.join("__screens/all", filename)
            )

            result["crawl_screen"] = filename

        except Exception as e:
            errors.append(f"screen_error: {repr(e)}")

        try:
            page_data = driver.execute_script(PAGE_SCRIPT)

            result["crawl_page_input"] = page_data.get("input", pd.NA)
            result["crawl_page_text"] = page_data.get("text", pd.NA)

        except Exception as e:
            errors.append(f"data_error: {repr(e)}")
            result["crawl_page_input"] = pd.NA
            result["crawl_page_text"] = pd.NA

    except Exception as e:
        errors.append(f"general_error: {repr(e)}")

    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass

    if errors:
        result["crawl_error"] = " | ".join(errors)

    return result_series(result)

In [15]:
%%time
# DEBUG
# res = check_url('http://www.vutbr.cz/',1)
# res.to_dict()

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 6.2 μs


## Main crawl

In [16]:
# Randomly select 1,000 records:
# data = data.sample(n=1000, random_state=0)

# Select the first 1,000 records:
# data = data.iloc[:1000]

# Select the last 1,000 records:
# data = data.iloc[-1000:]

# To process all records, do not subset data.

# Demonstration run:

data = data.sample(n=1000, random_state=0)

In [17]:
%%time

# Set the number of workers according to the available CPU cores and memory
pandarallel.initialize(progress_bar=True, nb_workers=50)

data[["crawl_final_url",
      "crawl_page_input",
      "crawl_page_text",
      "crawl_error",
      "crawl_date",
      "crawl_screen"]] = data.parallel_apply(lambda r: check_url(url=r.phishtank_url,phish_id=r.phishtank_id),axis=1)

INFO: Pandarallel will run on 50 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


CPU times: user 687 ms, sys: 165 ms, total: 852 ms
Wall time: 20min 38s


In [18]:
data.loc[data["crawl_screen"].notna()]

,phishtank_id,phishtank_url,phishtank_verified,phishtank_online,crawl_final_url,crawl_page_input,crawl_page_text,crawl_error,crawl_date,crawl_screen
13588,9420923,https://ledgr-live-faqs.wixstudio.com/collect,True,True,https://ledgr-live-faqs.wixstudio.com/collect,[],Design\nDevelopment\nBusiness solutions\nEnter...,<NA>,2026-08-19,https_ledgr-live-faqs.wixstudio.com_collect_94...
46122,8720219,https://qrco.de/bfJjYz,True,True,https://qrco.de/bfJjYz,[],"← Back to our website\nHey, thanks for scannin...",<NA>,2026-08-19,https_qrco.de_bfJjYz_8720219_cb1f601f6892.png
3798,9459107,https://kleinanzeigen.sfv3.com/355194501134?ma...,True,True,https://kleinanzeigen.sfv3.com/355194501134?ma...,[],Tento web není dostupný\n\nIP adresa serveru k...,<NA>,2026-08-19,https_kleinanzeigen.sfv3.com_355194501134_mail...
16826,9403699,https://willing-gift-685217.framer.app/,True,True,https://willing-gift-685217.framer.app/,[],Site Not Found\nThere is no site configured at...,<NA>,2026-08-19,https_willing-gift-685217.framer.app__9403699_...
14465,9416708,https://pzyiku.hmctmy.cn/ghticg/lub-iu_louik,True,True,https://pzyiku.hmctmy.cn/ghticg/lub-iu_louik,[],Tento web není dostupný\n\nWebové stránky na a...,<NA>,2026-08-19,https_pzyiku.hmctmy.cn_ghticg_lub-iu_louik_941...
...,...,...,...,...,...,...,...,...,...,...
50465,8647519,https://qrco.de/bfBcth,True,True,https://qrco.de/bfBcth,[],"← Back to our website\nHey, thanks for scannin...",<NA>,2026-08-19,https_qrco.de_bfBcth_8647519_291c45b1fdb2.png
29109,9261469,https://us12.campaign-archive.com/?u=348f58a0e...,True,True,https://mailchimp.com/about/mcsv-static,[],____\n / ___M ]__\nC{ ( o o )}\n { •...,<NA>,2026-08-19,https_mailchimp.com_about_mcsv-static_9261469_...
53491,8522543,https://happy-lovelace.5-100-224-131.plesk.page/,True,True,https://happy-lovelace.5-100-224-131.plesk.page/,"[{'autocomplete': '', 'disabled': False, 'heig...",Support Portal\nNederlands /\nEnglish /\nDeuts...,<NA>,2026-08-19,https_happy-lovelace.5-100-224-131.plesk.page_...
10674,9433235,https://www.ntxactu.com/,True,True,https://www.ntxactu.com/,[],Tento web není dostupný\n\nIP adresa serveru w...,<NA>,2026-08-19,https_www.ntxactu.com__9433235_a8a0de41bd97.png


In [19]:
data["crawl_error"].notna().mean()

np.float64(0.072)

In [20]:
todrop_crawl_error = data.loc[data.crawl_error.notna()]
display(len(todrop_crawl_error))
data = data.drop(todrop_crawl_error.index)

todrop_crawl_error

72

,phishtank_id,phishtank_url,phishtank_verified,phishtank_online,crawl_final_url,crawl_page_input,crawl_page_text,crawl_error,crawl_date,crawl_screen
62758,8080740,http://129.226.210.78/v3/signin/identifier?dsh...,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
8707,9441951,https://promociones-pc-ec-2026.gamer.gd,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
41733,8805297,https://qrco.de/bfTSip,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
44354,8745379,https://qrco.de/bfMu9p,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
21990,9344514,https://qrco.de/bgcXhF,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
...,...,...,...,...,...,...,...,...,...,...
64110,7955115,https://edavki-depozit.firebaseapp.com/,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
22544,9342187,https://qrco.de/0UTL00K,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
39784,8867944,https://easying-f4b8c.firebaseapp.com/,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN
51953,8596731,https://nnfems.com/Webmail/51/Webmail/webmail.php,True,True,NaN,NaN,NaN,no_rendered_text,2026-08-19,NaN


# Select structural candidates for manual review

In [21]:
def check_input(page_input,page_text):
    
    # Input types eligible for structural candidate screening
    eligible_input_types = {
        "text",
        "short_text",
        "password",
        "number",
        "email",
    }

    ignore_page_text = {"404",}

    page_text = page_text.lower()

    for word in ignore_page_text:
        if word in page_text: return False

    n_eligible_inputs = 0
    
    for item in page_input:

        #print(item)

        if not (item.get("visible") == True and 
                item.get("disabled") == False and 
                item.get("readonly") == False):
            continue                  
            
        input_type = item.get("type", "").lower()
        
        if not (input_type in eligible_input_types):
            continue
            
        #print(item)
        
        n_eligible_inputs = n_eligible_inputs + 1              
    
    if n_eligible_inputs >= 2: return True 
        
    return False

In [22]:
data['structural_candidate'] = data.apply(lambda r: check_input(page_input=r.crawl_page_input,page_text=r.crawl_page_text),axis=1)

data

,phishtank_id,phishtank_url,phishtank_verified,phishtank_online,crawl_final_url,crawl_page_input,crawl_page_text,crawl_error,crawl_date,crawl_screen,structural_candidate
13588,9420923,https://ledgr-live-faqs.wixstudio.com/collect,True,True,https://ledgr-live-faqs.wixstudio.com/collect,[],Design\nDevelopment\nBusiness solutions\nEnter...,<NA>,2026-08-19,https_ledgr-live-faqs.wixstudio.com_collect_94...,False
46122,8720219,https://qrco.de/bfJjYz,True,True,https://qrco.de/bfJjYz,[],"← Back to our website\nHey, thanks for scannin...",<NA>,2026-08-19,https_qrco.de_bfJjYz_8720219_cb1f601f6892.png,False
3798,9459107,https://kleinanzeigen.sfv3.com/355194501134?ma...,True,True,https://kleinanzeigen.sfv3.com/355194501134?ma...,[],Tento web není dostupný\n\nIP adresa serveru k...,<NA>,2026-08-19,https_kleinanzeigen.sfv3.com_355194501134_mail...,False
16826,9403699,https://willing-gift-685217.framer.app/,True,True,https://willing-gift-685217.framer.app/,[],Site Not Found\nThere is no site configured at...,<NA>,2026-08-19,https_willing-gift-685217.framer.app__9403699_...,False
14465,9416708,https://pzyiku.hmctmy.cn/ghticg/lub-iu_louik,True,True,https://pzyiku.hmctmy.cn/ghticg/lub-iu_louik,[],Tento web není dostupný\n\nWebové stránky na a...,<NA>,2026-08-19,https_pzyiku.hmctmy.cn_ghticg_lub-iu_louik_941...,False
...,...,...,...,...,...,...,...,...,...,...,...
50465,8647519,https://qrco.de/bfBcth,True,True,https://qrco.de/bfBcth,[],"← Back to our website\nHey, thanks for scannin...",<NA>,2026-08-19,https_qrco.de_bfBcth_8647519_291c45b1fdb2.png,False
29109,9261469,https://us12.campaign-archive.com/?u=348f58a0e...,True,True,https://mailchimp.com/about/mcsv-static,[],____\n / ___M ]__\nC{ ( o o )}\n { •...,<NA>,2026-08-19,https_mailchimp.com_about_mcsv-static_9261469_...,False
53491,8522543,https://happy-lovelace.5-100-224-131.plesk.page/,True,True,https://happy-lovelace.5-100-224-131.plesk.page/,"[{'autocomplete': '', 'disabled': False, 'heig...",Support Portal\nNederlands /\nEnglish /\nDeuts...,<NA>,2026-08-19,https_happy-lovelace.5-100-224-131.plesk.page_...,True
10674,9433235,https://www.ntxactu.com/,True,True,https://www.ntxactu.com/,[],Tento web není dostupný\n\nIP adresa serveru w...,<NA>,2026-08-19,https_www.ntxactu.com__9433235_a8a0de41bd97.png,False


In [23]:
todrop_no_structural_candidate = data.loc[data.structural_candidate == False]
display(len(todrop_no_structural_candidate))
data = data.drop(todrop_no_structural_candidate.index)

todrop_no_structural_candidate.apply(
    lambda r: copy_screen_meta(r,'__screens/structural-candidate-no'),axis=1)

todrop_no_structural_candidate

855

,phishtank_id,phishtank_url,phishtank_verified,phishtank_online,crawl_final_url,crawl_page_input,crawl_page_text,crawl_error,crawl_date,crawl_screen,structural_candidate
13588,9420923,https://ledgr-live-faqs.wixstudio.com/collect,True,True,https://ledgr-live-faqs.wixstudio.com/collect,[],Design\nDevelopment\nBusiness solutions\nEnter...,<NA>,2026-08-19,https_ledgr-live-faqs.wixstudio.com_collect_94...,False
46122,8720219,https://qrco.de/bfJjYz,True,True,https://qrco.de/bfJjYz,[],"← Back to our website\nHey, thanks for scannin...",<NA>,2026-08-19,https_qrco.de_bfJjYz_8720219_cb1f601f6892.png,False
3798,9459107,https://kleinanzeigen.sfv3.com/355194501134?ma...,True,True,https://kleinanzeigen.sfv3.com/355194501134?ma...,[],Tento web není dostupný\n\nIP adresa serveru k...,<NA>,2026-08-19,https_kleinanzeigen.sfv3.com_355194501134_mail...,False
16826,9403699,https://willing-gift-685217.framer.app/,True,True,https://willing-gift-685217.framer.app/,[],Site Not Found\nThere is no site configured at...,<NA>,2026-08-19,https_willing-gift-685217.framer.app__9403699_...,False
14465,9416708,https://pzyiku.hmctmy.cn/ghticg/lub-iu_louik,True,True,https://pzyiku.hmctmy.cn/ghticg/lub-iu_louik,[],Tento web není dostupný\n\nWebové stránky na a...,<NA>,2026-08-19,https_pzyiku.hmctmy.cn_ghticg_lub-iu_louik_941...,False
...,...,...,...,...,...,...,...,...,...,...,...
43126,8771310,https://docs.google.com/presentation/d/1_lnMAm...,True,True,https://docs.google.com/presentation/d/1_lnMAm...,[],1\nChcete-li aktivovat podporu čtečky obrazovk...,<NA>,2026-08-19,https_docs.google.com_presentation_d_1_lnMAmuF...,False
50465,8647519,https://qrco.de/bfBcth,True,True,https://qrco.de/bfBcth,[],"← Back to our website\nHey, thanks for scannin...",<NA>,2026-08-19,https_qrco.de_bfBcth_8647519_291c45b1fdb2.png,False
29109,9261469,https://us12.campaign-archive.com/?u=348f58a0e...,True,True,https://mailchimp.com/about/mcsv-static,[],____\n / ___M ]__\nC{ ( o o )}\n { •...,<NA>,2026-08-19,https_mailchimp.com_about_mcsv-static_9261469_...,False
10674,9433235,https://www.ntxactu.com/,True,True,https://www.ntxactu.com/,[],Tento web není dostupný\n\nIP adresa serveru w...,<NA>,2026-08-19,https_www.ntxactu.com__9433235_a8a0de41bd97.png,False


In [24]:
data.apply(lambda r: copy_screen_meta(r,'__screens/structural-candidate-yes'),axis=1)

data

,phishtank_id,phishtank_url,phishtank_verified,phishtank_online,crawl_final_url,crawl_page_input,crawl_page_text,crawl_error,crawl_date,crawl_screen,structural_candidate
25001,9330459,https://mailboxupdate0.weebly.com/,True,True,https://mailboxupdate0.weebly.com/,"[{'autocomplete': '', 'disabled': False, 'heig...",\tMAIL UPDATE\t\nHome\nServices\nAbout\nNews\n...,<NA>,2026-08-19,https_mailboxupdate0.weebly.com__9330459_9a8c1...,True
50808,8635870,https://upgradedmono.pages.dev/,True,True,https://upgradedmono.pages.dev/?websrc=DgHgaS0...,"[{'autocomplete': '', 'disabled': False, 'heig...",Email:\nPassword:\n\nThis information system i...,<NA>,2026-08-19,https_upgradedmono.pages.dev_websrc_DgHgaS089A...,True
36848,9020830,https://ophtyfg.weebly.com/,True,True,https://ophtyfg.weebly.com/,"[{'autocomplete': '', 'disabled': False, 'heig...",\t\n\t\nEmail address *\nPassw**d *\nNEXT\n\t...,<NA>,2026-08-19,https_ophtyfg.weebly.com__9020830_00b538773ac0...,True
48567,8688986,https://pub-3a558ddf24154724bfb87a3d02066114.r...,True,True,https://pub-3a558ddf24154724bfb87a3d02066114.r...,"[{'autocomplete': '', 'disabled': False, 'heig...",English\nالعربية\nбългарски\nবাংলা\nCatalà\nČe...,<NA>,2026-08-19,https_pub-3a558ddf24154724bfb87a3d02066114.r2....,True
7837,9445275,https://affable-subtasks-609343.framer.app/,True,True,https://affable-subtasks-609343.framer.app/,"[{'autocomplete': '', 'disabled': False, 'heig...",Se connecter\n\nCreate a free website with Fra...,<NA>,2026-08-19,https_affable-subtasks-609343.framer.app__9445...,True
...,...,...,...,...,...,...,...,...,...,...,...
21161,9355045,https://docs.google.com/forms/d/e/1FAIpQLSfNMa...,True,True,https://docs.google.com/forms/d/e/1FAIpQLSfNMa...,"[{'autocomplete': 'off', 'disabled': False, 'h...",Important Form\nKindly fill up the fields corr...,<NA>,2026-08-19,https_docs.google.com_forms_d_e_1FAIpQLSfNManw...,True
28711,9272465,https://big750530.wixsite.com/my-site-1,True,True,https://big750530.wixsite.com/my-site-1,"[{'autocomplete': '', 'disabled': False, 'heig...",This website was built on Wix. Create yours to...,<NA>,2026-08-19,https_big750530.wixsite.com_my-site-1_9272465_...,True
24839,9333069,https://capesminecraftbr.weebly.com/,True,True,https://capesminecraftbr.weebly.com/,"[{'autocomplete': '', 'disabled': False, 'heig...",\tMinecraft Capes BR\t\nHome\nAbout\nBlog\nCon...,<NA>,2026-08-19,https_capesminecraftbr.weebly.com__9333069_15b...,True
36527,9030339,https://collinscu.weebly.com/,True,True,https://collinscu.weebly.com/,"[{'autocomplete': '', 'disabled': False, 'heig...",\t\nMENU\nUser ID\nPassword\nRemember my User...,<NA>,2026-08-19,https_collinscu.weebly.com__9030339_2cbf253d1e...,True
